In [0]:
from datetime import datetime
from pyspark.sql import Row
from uuid import uuid4

def init_logs_table():
    
    ####### spark.sql("DROP TABLE IF EXISTS training.monitoring.pipeline_logs")

    spark.sql("CREATE SCHEMA IF NOT EXISTS training.monitoring")

    spark.sql("""
    CREATE TABLE IF NOT EXISTS training.monitoring.pipeline_logs
    (   run_id STRING,
        run_timestamp TIMESTAMP,
        step STRING,
        status STRING,
        row_count BIGINT,
        message STRING
    )
    USING DELTA
    """)

init_logs_table()


def write_log(step, status, row_count, message):
    spark.createDataFrame([
        Row(
            run_id=str(uuid4()),
            run_timestamp=datetime.now(),
            step=step,
            status=status,
            row_count=row_count,
            message=message
        )
    ]).write \
      .mode("append") \
      .saveAsTable("training.monitoring.pipeline_logs")
      
path = "/Volumes/training/bronze/sales_volume/orders.csv"

print("=" * 100)
print("BRONZE START")
print("=" * 100)

try:
    df = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(path)
    )

    print(f"Bronze path = {path}")
    print(f"df.columns = {df.columns}")

    # Nettoyage des noms de colonnes
    df = df.toDF(*[
            c.strip().lower().replace(" ", "") 
            for c in df.columns
            ])

    print("correction des espaces sur le nom des champs")
    print(f"df.columns = {df.columns}")

    row_count = df.count()

    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(
          "training.bronze.orders_raw"
      )

    print("Ecriture dans la table training.bronze.orders_raw")

    write_log(
        step="BRONZE",
        status="SUCCESS",
        row_count=row_count,
        message="Chargement Bronze terminé"
    )

except Exception as e:

    write_log(
        step="BRONZE",
        status="FAILED",
        row_count=0,
        message=str(e)
    )

    raise

print("=" * 50)
print("BRONZE END")
print("=" * 50)



In [0]:

spark.sql("SELECT * FROM training.monitoring.pipeline_logs ORDER BY run_timestamp ").show()